# TP2 : Extraction d’information

Ce notebook présente l’analyse de 100 fichiers du corpus CAMille pour l’année 1961. Il regroupe l’extraction de mots-clés, la création d’un nuage de mots, le repérage d’entités nommées et l’analyse de sentiment de dix extraits.

## Corpus et année étudiée

Les fichiers texte doivent se trouver dans `data/txt` à la racine du dépôt. Le code retrouve ce dossier à partir de l’emplacement courant du notebook.

In [ ]:
from pathlib import Path

year = 1961

current_dir = Path.cwd().resolve()
repo_root = next(
    (folder for folder in (current_dir, *current_dir.parents)
     if (folder / "data" / "txt").is_dir()),
    None
)

if repo_root is None:
    raise FileNotFoundError(
        f"Le dossier data/txt est introuvable depuis {current_dir}."
    )

txt_path = repo_root / "data" / "txt"
files_year = sorted(txt_path.glob(f"*{year}*.txt"))

if not files_year:
    raise FileNotFoundError(f"Aucun fichier trouvé pour l’année {year}.")

print(f"Fichiers trouvés pour {year} : {len(files_year)}")

## 1. Extraction des mots-clés

YAKE extrait des expressions de deux mots dans chaque fichier. Le comptage indique dans combien de fichiers chaque expression apparaît ; une expression n’est comptée qu’une fois par fichier.

In [ ]:
from collections import Counter
import yake

kw_extractor = yake.KeywordExtractor(lan="fr", top=50)
keyword_counts = Counter()

for file_path in files_year:
    text = file_path.read_text(encoding="utf-8")
    keywords = kw_extractor.extract_keywords(text)

    # Une expression ne compte qu’une fois dans chaque fichier.
    bigrams = {
        keyword.lower()
        for keyword, score in keywords
        if len(keyword.split()) == 2
    }
    keyword_counts.update(bigrams)

for keyword, count in keyword_counts.most_common(30):
    print(f"{keyword} : présent dans {count} fichiers")

### Observation

Les expressions les plus fréquentes comprennent « agence Rossel » (16 fichiers), « New York » (15) et « écrire agence » (11). Les résultats mêlent références à des organisations, lieux et annonces. Certaines expressions semblent aussi contenir des abréviations ou des erreurs d’OCR.

## 2. Nuage de mots

Les textes de 1961 sont regroupés, puis les mots fréquents sont filtrés à l’aide de stopwords. La liste a été enrichie en plusieurs essais pour écarter notamment des mots de formulaire et des fragments OCR, tout en gardant des termes de contenu comme les noms de lieux.

In [ ]:
from collections import Counter
from IPython.display import Image
from nltk.corpus import stopwords
from wordcloud import WordCloud
import nltk

nltk.download("stopwords", quiet=True)

sw = set(stopwords.words("french"))
sw.update([
    "agence", "rossel", "ecrire", "app", "tél",
    "rue", "plus", "tout", "dem", "ans", "cette", "deux",
    "sans", "bien", "après", "très", "être", "comme",
    "sous", "soir", "van", "brux", "fait", "tous", "bon",
    "dont", "ecr", "faire", "part", "entre", "non",
    "demande", "aussi", "bonne", "grand", "contre", "peu",
    "encore", "trois", "mois", "mod", "gar", "vis"
])

text_year = " ".join(
    file_path.read_text(encoding="utf-8")
    for file_path in files_year
)
tokens = nltk.wordpunct_tokenize(text_year)
clean_words = [
    word.upper()
    for word in tokens
    if len(word) > 2 and word.isalpha() and word.lower() not in sw
]

frequencies = Counter(clean_words)
print(frequencies.most_common(20))

tmp_path = repo_root / "data" / "tmp"
tmp_path.mkdir(parents=True, exist_ok=True)
image_path = tmp_path / f"{year}.png"

cloud = WordCloud(
    width=2000,
    height=1000,
    background_color="white"
).generate_from_frequencies(frequencies)

cloud.to_file(str(image_path))
Image(filename=str(image_path))

### Observation

Le nuage fait ressortir des lieux et des termes associés aux annonces. L’ajout des stopwords réduit la place des formules répétitives, mais des formes liées à l’OCR restent visibles. L’image est enregistrée dans `data/tmp/1961.png` pour pouvoir être insérée dans le rapport.

## 3. Reconnaissance d’entités nommées

SpaCy est utilisé pour repérer les personnes, les organisations et les lieux dans les fichiers de 1961. Les fréquences sont calculées sur l’ensemble des textes sélectionnés.

In [ ]:
from collections import defaultdict
import spacy

nlp = spacy.load("fr_core_news_md")

entities = {
    "Personnes": defaultdict(int),
    "Organisations": defaultdict(int),
    "Lieux": defaultdict(int),
}

texts = (
    file_path.read_text(encoding="utf-8")
    for file_path in files_year
)

for doc in nlp.pipe(texts, batch_size=16):
    for ent in doc.ents:
        name = ent.text.strip()
        if len(name) <= 3:
            continue
        if ent.label_ == "PER":
            entities["Personnes"][name] += 1
        elif ent.label_ == "ORG":
            entities["Organisations"][name] += 1
        elif ent.label_ in ("LOC", "GPE"):
            entities["Lieux"][name] += 1

for category, counts in entities.items():
    print(f"\n{category} :")
    top_five = sorted(
        counts.items(),
        key=lambda item: item[1],
        reverse=True
    )[:5]
    for name, count in top_five:
        print(f"{name} : {count}")

### Observation

Les lieux les plus fréquents sont Bruxelles (464 mentions), Anvers (218) et Liège (151). Les catégories automatiques contiennent aussi des formes discutables : « Téléph » est classé comme lieu, tandis que « SENT » et « Ecrire Ag » apparaissent parmi les personnes. Ces résultats sont à interpréter avec prudence, car l’OCR et le classement de SpaCy peuvent produire des erreurs.

## 4. Analyse de sentiment

Dix extraits de 1961 ont été retenus. Les graphies OCR sont conservées dans les phrases afin d’analyser le texte tel qu’il apparaît dans le corpus. TextBlob-FR calcule pour chaque extrait une polarité et une subjectivité.

In [ ]:
import pandas as pd
from textblob import Blobber
from textblob_fr import PatternTagger, PatternAnalyzer

tb = Blobber(
    pos_tagger=PatternTagger(),
    analyzer=PatternAnalyzer()
)

def get_sentiment(text):
    return tb(text).sentiment

selected_sentences = [
    {
        "Fichier": "KB_JB838_1961-03-08_01-00012.txt",
        "Phrase": "Selon le même rapport, il y aurait eu ensuite un grondement de tonnerre, puis tout aurait sombré dans le silence..."
    },
    {
        "Fichier": "KB_JB838_1961-12-15_01-00024.txt",
        "Phrase": "Milton Ribeiro a gardé, dans le rôle du chef des bandits, quelque chose de sa sauvagerie spontanée, de son autorité foudroyante."
    },
    {
        "Fichier": "KB_JB838_1961-06-14_01-00012.txt",
        "Phrase": "Quand l'insecte marche, il se sert de ses deux « phares » de pointe."
    },
    {
        "Fichier": "KB_JB838_1961-02-04_01-00004.txt",
        "Phrase": "Cependant, lu physionomie générale du marché ne subit pas de changement appréciable, les molns-value de fractions restant en majorité."
    },
    {
        "Fichier": "KB_JB838_1961-12-15_01-00024.txt",
        "Phrase": "Traqué par la police, le tueur d’enfants l’est aussi ijpr la toute-puissante organisation des clochards, malfaiteurs et mendiants."
    },
    {
        "Fichier": "KB_JB838_1961-07-01_01-00011.txt",
        "Phrase": "Aimeriez- vous aller danser ce soir 7 l^es yeux de Glna brillèrent de plaisir."
    },
    {
        "Fichier": "KB_JB838_1961-09-08_01-00024.txt",
        "Phrase": "Dany Cartel et un Georges Bras' sens peu à l'aise figurent également au générique d'un film d'auteur qui a gardé son alacrité, sa fantaisie et sa tendresse."
    },
    {
        "Fichier": "KB_JB838_1961-06-16_01-00024.txt",
        "Phrase": "Quant a la thèse soutenue dans l’œuvre, elle s’identifie aux prémisses et constitue en quelque sorte un problème résolu d’avance."
    },
    {
        "Fichier": "KB_JB838_1961-08-02_01-00017.txt",
        "Phrase": "C’est un fuit médical bien connu ; l'été est la période idéale pour suivre un traitement des cheveux et du cuir chevelu."
    },
    {
        "Fichier": "KB_JB838_1961-12-26_01-00003.txt",
        "Phrase": "Vous n'avez pas résolu ce pro blètne ni rempli vos engagements Pourtant le projet était réalisable La vérité est que le comité central du parti communiste ukrainien et le conseil des ministres de l’Ukraine se sont révélés incapables."
    },
]

if len(selected_sentences) != 10:
    raise ValueError("Le tableau doit contenir exactement dix extraits.")

### Résultats des dix extraits

La cellule suivante recalcule les scores à partir de la liste finale ci-dessus. Le tableau doit être relu après exécution, car certains extraits contiennent des erreurs d’OCR.

In [ ]:
results = []

for item in selected_sentences:
    polarity, subjectivity = get_sentiment(item["Phrase"])
    results.append({
        "Fichier": item["Fichier"],
        "Phrase": item["Phrase"],
        "Polarité": round(polarity, 3),
        "Subjectivité": round(subjectivity, 3),
    })

sentiment_table = pd.DataFrame(results)
pd.set_option("display.max_colwidth", None)
sentiment_table

### Résumé des scores

Les nombres de polarités positives, nulles et négatives, ainsi que les moyennes, sont calculés à partir du tableau final. Ils doivent être repris dans le rapport après exécution.

In [ ]:
polarities = sentiment_table["Polarité"].astype(float)
subjectivities = sentiment_table["Subjectivité"].astype(float)

print("Polarités positives :", (polarities > 0).sum())
print("Polarités nulles :", (polarities == 0).sum())
print("Polarités négatives :", (polarities < 0).sum())
print("Polarité moyenne :", round(polarities.mean(), 3))
print("Subjectivité moyenne :", round(subjectivities.mean(), 3))

### Observation

Sur les dix extraits retenus, sept ont une polarité positive, deux une polarité nulle et un une polarité négative. La polarité moyenne est d’environ 0,091 et la subjectivité moyenne d’environ 0,212. La polarité moyenne reste proche de zéro. Ces résultats portent uniquement sur les dix extraits choisis ; les erreurs d’OCR peuvent aussi influencer les scores.